# Toxic Comment Filtering - Local Training Notebook

Notebook này dùng để:
- đọc dữ liệu ViCTSD từ thư mục `data/`
- train Logistic Regression
- fine-tune PhoBERT
- lưu model vào `models/`
- lưu metrics vào `reports/`

Notebook được viết để chạy local hoặc trên VS Code/Jupyter, không dùng đường dẫn `/content/...`.


In [1]:
from pathlib import Path
import json
import re
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.pipeline import Pipeline
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings('ignore')

ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / 'data'
MODELS_DIR = ROOT_DIR / 'models'
REPORTS_DIR = ROOT_DIR / 'reports'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT_DIR =', ROOT_DIR)
print('DATA_DIR =', DATA_DIR)
print('MODELS_DIR =', MODELS_DIR)
print('REPORTS_DIR =', REPORTS_DIR)
print('CUDA available =', torch.cuda.is_available())


c:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT_DIR = c:\Users\ACER\Downloads\toxic_comment_project_persistent_ready\toxic_comment_project
DATA_DIR = c:\Users\ACER\Downloads\toxic_comment_project_persistent_ready\toxic_comment_project\data
MODELS_DIR = c:\Users\ACER\Downloads\toxic_comment_project_persistent_ready\toxic_comment_project\models
REPORTS_DIR = c:\Users\ACER\Downloads\toxic_comment_project_persistent_ready\toxic_comment_project\reports
CUDA available = True


In [2]:
def normalize_text(text: str) -> str:
    text = '' if pd.isna(text) else str(text)
    text = text.lower().strip()
    text = re.sub(r'https?://\\S+|www\\.\\S+', ' <url> ', text)
    text = re.sub(r'\\S+@\\S+', ' <email> ', text)
    text = re.sub(r'@[\\w_]+', ' <user> ', text)
    text = re.sub(r'\\d{8,}', ' <number> ', text)
    text = re.sub(r'([!?.]){2,}', r' \\1 ', text)
    text = re.sub(r'(.)\\1{3,}', r'\\1\\1\\1', text)
    text = re.sub(r'\\s+', ' ', text).strip()
    return text


def load_split(name: str) -> pd.DataFrame:
    path = DATA_DIR / f'ViCTSD_{name}.csv'
    df = pd.read_csv(path)
    df = df.rename(columns={
        'Comment': 'comment',
        'Toxicity': 'label',
        'Title': 'title',
        'Topic': 'topic',
    })
    df['comment'] = df['comment'].astype(str)
    df['label'] = df['label'].astype(int)
    df['clean_text'] = df['comment'].map(normalize_text)
    return df


train_df = load_split('train')
valid_df = load_split('valid')
test_df = load_split('test')

print(train_df.shape, valid_df.shape, test_df.shape)
display(train_df.head())


(7000, 7) (2000, 7) (1000, 7)


,Unnamed: 0,comment,Constructiveness,label,title,topic,clean_text
0,6326,Thật tuyệt vời...!!!,0,0,Những 'bước tiến diệu kỳ' của Trúc Nhi - Diệu Nhi,SucKhoe,thật tuyệt vời \1
1,7835,"mỹ đã tuột dốc quá nhiều rồi, giờ muốn vực dậy...",1,0,Hình tượng Mỹ sụp đổ trong lòng người dân thế ...,TheGioi,"mỹ đã tuột dốc quá nhiều rồi, giờ muốn vực dậy..."
2,4690,tôi thấy người lái xe hơi bấm còi mới là người...,1,1,Cả trăm người đạp xe thể dục bịt kín đường,OtoXemay,tôi thấy người lái xe hơi bấm còi mới là người...
3,6011,Coi dịch là giặc. Đã mang tên đó mà xâm nhập V...,0,0,11 ngày không lây nhiễm nCoV cộng đồng,SucKhoe,coi dịch là giặc. đã mang tên đó mà xâm nhập v...
4,9303,Thương các bé quá! Các con còn quá nhỏ mà đã p...,0,0,5 trẻ chết đuối dưới ao,ThoiSu,thương các bé quá! các con còn quá nhỏ mà đã p...


## 1. Train Logistic Regression


In [3]:
logreg_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )),
    ('clf', LogisticRegression(
        max_iter=1200,
        class_weight='balanced',
        solver='liblinear',
        random_state=42,
    )),
])

logreg_pipeline.fit(train_df['clean_text'], train_df['label'])

logreg_output_path = MODELS_DIR / 'logreg_toxic_pipeline.joblib'
joblib.dump(logreg_pipeline, logreg_output_path)
print('Saved Logistic Regression to:', logreg_output_path)


Saved Logistic Regression to: c:\Users\ACER\Downloads\toxic_comment_project_persistent_ready\toxic_comment_project\models\logreg_toxic_pipeline.joblib


In [4]:
logreg_probs = logreg_pipeline.predict_proba(test_df['clean_text'])[:, 1]
logreg_preds = (logreg_probs >= 0.5).astype(int)
true_labels = test_df['label'].to_numpy()

acc = accuracy_score(true_labels, logreg_preds)
prec, rec, f1, _ = precision_recall_fscore_support(
    true_labels,
    logreg_preds,
    average='binary',
    zero_division=0,
)
report = classification_report(
    true_labels,
    logreg_preds,
    target_names=['clean', 'toxic'],
    output_dict=True,
    zero_division=0,
)
cm = confusion_matrix(true_labels, logreg_preds)

logreg_metrics = {
    'accuracy': float(acc),
    'precision_toxic': float(prec),
    'recall_toxic': float(rec),
    'f1_toxic': float(f1),
    'classification_report': report,
    'threshold': 0.5,
    'model_name': 'Logistic Regression',
}

with open(REPORTS_DIR / 'logreg_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(logreg_metrics, f, ensure_ascii=False, indent=2)

logreg_pred_df = test_df[['comment', 'title', 'topic', 'label']].copy()
logreg_pred_df['true_label'] = logreg_pred_df['label']
logreg_pred_df['pred_label'] = logreg_preds
logreg_pred_df['prob_toxic'] = logreg_probs
logreg_pred_df.drop(columns=['label'], inplace=True)
logreg_pred_df.to_csv(REPORTS_DIR / 'logreg_predictions.csv', index=False)

error_df = logreg_pred_df[logreg_pred_df['true_label'] != logreg_pred_df['pred_label']].copy()
error_df.to_csv(REPORTS_DIR / 'logreg_errors.csv', index=False)

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['clean', 'toxic'])
ax.set_yticklabels(['clean', 'toxic'])
ax.set_title('LogReg Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
for i in range(2):
    for j in range(2):
        ax.text(j, i, int(cm[i, j]), ha='center', va='center')
fig.tight_layout()
fig.savefig(REPORTS_DIR / 'logreg_confusion_matrix.png', dpi=160)
plt.close(fig)

print(logreg_metrics)


{'accuracy': 0.859, 'precision_toxic': 0.40372670807453415, 'recall_toxic': 0.5909090909090909, 'f1_toxic': 0.4797047970479705, 'classification_report': {'clean': {'precision': 0.9463647199046484, 'recall': 0.8921348314606742, 'f1-score': 0.91844997108155, 'support': 890.0}, 'toxic': {'precision': 0.40372670807453415, 'recall': 0.5909090909090909, 'f1-score': 0.4797047970479705, 'support': 110.0}, 'accuracy': 0.859, 'macro avg': {'precision': 0.6750457139895913, 'recall': 0.7415219611848826, 'f1-score': 0.6990773840647603, 'support': 1000.0}, 'weighted avg': {'precision': 0.8866745386033359, 'recall': 0.859, 'f1-score': 0.8701880019378563, 'support': 1000.0}}, 'threshold': 0.5, 'model_name': 'Logistic Regression'}


## 2. Fine-tune PhoBERT

Bạn có thể đặt `FAST_DEV_RUN = True` để test nhanh. Khi chạy thật thì để `False`.


In [5]:
MODEL_CHECKPOINT = 'vinai/phobert-base-v2'
FAST_DEV_RUN = False
NUM_EPOCHS = 1 if FAST_DEV_RUN else 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=False)

hf_train = Dataset.from_pandas(train_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'}), preserve_index=False)
hf_valid = Dataset.from_pandas(valid_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'}), preserve_index=False)
hf_test = Dataset.from_pandas(test_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'}), preserve_index=False)

def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128)

hf_train = hf_train.map(tokenize_batch, batched=True)
hf_valid = hf_valid.map(tokenize_batch, batched=True)
hf_test = hf_test.map(tokenize_batch, batched=True)

hf_train = hf_train.remove_columns(['text'])
hf_valid = hf_valid.remove_columns(['text'])
hf_test = hf_test.remove_columns(['text'])

hf_train.set_format('torch')
hf_valid.set_format('torch')
hf_test.set_format('torch')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    return {
        'accuracy': float(acc),
        'precision': float(prec),
        'recall': float(rec),
        'f1': float(f1),
    }

phobert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)

training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / 'phobert_checkpoints'),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
)

trainer = Trainer(
    model=phobert_model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


Map: 100%|██████████| 1000/1000 [00:00<00:00, 10638.10 examples/s]
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  3%|▎         | 51/1750 [00:11<03:34,  7.91it/s] 

{'loss': 0.3937, 'grad_norm': 4.845047473907471, 'learning_rate': 1.942857142857143e-05, 'epoch': 0.06}


  6%|▌         | 101/1750 [00:18<03:25,  8.01it/s]

{'loss': 0.3998, 'grad_norm': 1.2773873805999756, 'learning_rate': 1.885714285714286e-05, 'epoch': 0.11}


  9%|▊         | 152/1750 [00:24<03:19,  8.01it/s]

...

                                                   
100%|██████████| 1750/1750 [04:03<00:00,  7.28it/s]

{'eval_loss': 0.32719066739082336, 'eval_accuracy': 0.908, 'eval_precision': 0.6818181818181818, 'eval_recall': 0.3879310344827586, 'eval_f1': 0.4945054945054945, 'eval_runtime': 4.8242, 'eval_samples_per_second': 414.574, 'eval_steps_per_second': 51.822, 'epoch': 2.0}


100%|██████████| 1750/1750 [04:05<00:00,  7.14it/s]

{'train_runtime': 245.1351, 'train_samples_per_second': 57.111, 'train_steps_per_second': 7.139, 'train_loss': 0.2843534573146275, 'epoch': 2.0}


TrainOutput(global_step=1750, training_loss=0.2843534573146275, metrics={'train_runtime': 245.1351, 'train_samples_per_second': 57.111, 'train_steps_per_second': 7.139, 'total_flos': 618508313387520.0, 'train_loss': 0.2843534573146275, 'epoch': 2.0})

In [6]:
phobert_output_dir = MODELS_DIR / 'phobert_toxic_model'
phobert_output_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(phobert_output_dir))
tokenizer.save_pretrained(str(phobert_output_dir))
print('Saved PhoBERT to:', phobert_output_dir)

raw_preds = trainer.predict(hf_test)
logits = raw_preds.predictions
phobert_preds = np.argmax(logits, axis=-1)
phobert_probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
true_labels = raw_preds.label_ids

acc = accuracy_score(true_labels, phobert_preds)
prec, rec, f1, _ = precision_recall_fscore_support(
    true_labels,
    phobert_preds,
    average='binary',
    zero_division=0,
)
report = classification_report(
    true_labels,
    phobert_preds,
    target_names=['clean', 'toxic'],
    output_dict=True,
    zero_division=0,
)
cm = confusion_matrix(true_labels, phobert_preds)

phobert_metrics = {
    'accuracy': float(acc),
    'precision_toxic': float(prec),
    'recall_toxic': float(rec),
    'f1_toxic': float(f1),
    'classification_report': report,
    'threshold': 0.5,
    'model_name': 'PhoBERT',
}

with open(REPORTS_DIR / 'phobert_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(phobert_metrics, f, ensure_ascii=False, indent=2)

phobert_pred_df = test_df[['comment', 'title', 'topic', 'label']].copy()
phobert_pred_df['true_label'] = phobert_pred_df['label']
phobert_pred_df['pred_label'] = phobert_preds
phobert_pred_df['prob_toxic'] = phobert_probs
phobert_pred_df.drop(columns=['label'], inplace=True)
phobert_pred_df.to_csv(REPORTS_DIR / 'phobert_predictions.csv', index=False)

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['clean', 'toxic'])
ax.set_yticklabels(['clean', 'toxic'])
ax.set_title('PhoBERT Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
for i in range(2):
    for j in range(2):
        ax.text(j, i, int(cm[i, j]), ha='center', va='center')
fig.tight_layout()
fig.savefig(REPORTS_DIR / 'phobert_confusion_matrix.png', dpi=160)
plt.close(fig)

print(phobert_metrics)


Saved PhoBERT to: c:\Users\ACER\Downloads\toxic_comment_project_persistent_ready\toxic_comment_project\models\phobert_toxic_model


100%|██████████| 125/125 [00:02<00:00, 52.45it/s]


{'accuracy': 0.909, 'precision_toxic': 0.6301369863013698, 'recall_toxic': 0.41818181818181815, 'f1_toxic': 0.5027322404371585, 'classification_report': {'clean': {'precision': 0.9309600862998921, 'recall': 0.9696629213483146, 'f1-score': 0.9499174463401211, 'support': 890.0}, 'toxic': {'precision': 0.6301369863013698, 'recall': 0.41818181818181815, 'f1-score': 0.5027322404371585, 'support': 110.0}, 'accuracy': 0.909, 'macro avg': {'precision': 0.780548536300631, 'recall': 0.6939223697650664, 'f1-score': 0.7263248433886398, 'support': 1000.0}, 'weighted avg': {'precision': 0.8978695453000546, 'recall': 0.909, 'f1-score': 0.9007270736907951, 'support': 1000.0}}, 'threshold': 0.5, 'model_name': 'PhoBERT'}


## 3. Kiểm tra file đầu ra


In [7]:
print('MODELS:')
for p in sorted(MODELS_DIR.rglob('*')):
    print(' -', p.relative_to(ROOT_DIR))

print('\nREPORTS:')
for p in sorted(REPORTS_DIR.rglob('*')):
    print(' -', p.relative_to(ROOT_DIR))


MODELS:
 - models\logreg_toxic_pipeline.joblib
 - models\phobert_checkpoints
 - models\phobert_checkpoints\checkpoint-1750
 - models\phobert_checkpoints\checkpoint-1750\added_tokens.json
 - models\phobert_checkpoints\checkpoint-1750\bpe.codes
 - models\phobert_checkpoints\checkpoint-1750\config.json
 - models\phobert_checkpoints\checkpoint-1750\model.safetensors
 - models\phobert_checkpoints\checkpoint-1750\optimizer.pt
 - models\phobert_checkpoints\checkpoint-1750\rng_state.pth
 - models\phobert_checkpoints\checkpoint-1750\scheduler.pt
 - models\phobert_checkpoints\checkpoint-1750\special_tokens_map.json
 - models\phobert_checkpoints\checkpoint-1750\tokenizer_config.json
 - models\phobert_checkpoints\checkpoint-1750\trainer_state.json
 - models\phobert_checkpoints\checkpoint-1750\training_args.bin
 - models\phobert_checkpoints\checkpoint-1750\vocab.txt
 - models\phobert_checkpoints\checkpoint-875
 - models\phobert_checkpoints\checkpoint-875\added_tokens.json
 - models\phobert_checkpoi

Sau khi chạy xong notebook, app sẽ dùng các file này:

- `models/logreg_toxic_pipeline.joblib`
- `models/phobert_toxic_model/`
- `reports/logreg_metrics.json`
- `reports/logreg_predictions.csv`
- `reports/logreg_confusion_matrix.png`
- `reports/phobert_metrics.json`
- `reports/phobert_predictions.csv`
- `reports/phobert_confusion_matrix.png`
